# 1 - Configurações, Bibliotecas e Funções Auxiliares

In [ ]:
# ==========================================
# 3 - Analítico - Extração do Histórico (Camada Bronze)
# ==========================================
from datetime import datetime, timedelta
import pandas as pd
import os
import requests

def extrair_historico(data_inicio, data_fim):
    token = obter_access_token()
    pagina = 1
    todos_atendimentos = []
    registros_anteriores = None
    url = f"https://api.mktzap.com.br/company/{COMPANY_ID}/historycontact"
    params = {"createdFrom": data_inicio, "createdTo": data_fim}

    print(f"⏳ Extraindo histórico de {data_inicio} até {data_fim}...")
    while True:
        if pagina > 500: break
        headers = {**HEADERS_PADRAO, "Authorization": f"Bearer {token}", "x-page": str(pagina)}
        res = requests.get(url, headers=headers, params=params, verify=False, timeout=30)
        
        if res.status_code == 401:
            token = obter_access_token()
            continue
            
        dados = res.json()
        if not dados or dados == registros_anteriores: break
        
        registros_anteriores = dados
        todos_atendimentos.extend(dados)
        if pagina % 10 == 0: print(f"  -> {pagina} páginas processadas...")
        pagina += 1
        
    return todos_atendimentos

# Lógica de Datas D-1
hoje = datetime.now()
ontem = hoje - timedelta(days=1)

primeiro_dia = ontem.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
ultimo_dia = ontem.replace(hour=23, minute=59, second=59, microsecond=0)

created_from = primeiro_dia.strftime("%Y-%m-%dT%H:%M:%S-03:00")
created_to = ultimo_dia.strftime("%Y-%m-%dT%H:%M:%S-03:00")

dados_brutos = extrair_historico(created_from, created_to)
df_bronze = pd.DataFrame(dados_brutos)
print(f"✅ Extração bruta concluída! Total: {len(df_bronze)} registros.")

if not df_bronze.empty:
    pasta_mes = ontem.strftime("%Y%m")
    pasta_27 = os.path.join(os.getcwd(), "documentos", pasta_mes, "27 - Brasil")
    os.makedirs(pasta_27, exist_ok=True)
    
    nome_bruto = f"historico_atendimentos_periodo_{primeiro_dia.strftime('%Y%m%d')}_a_{ultimo_dia.strftime('%Y%m%d')}.csv"
    caminho_bruto = os.path.join(pasta_27, nome_bruto)
    df_bronze.to_csv(caminho_bruto, sep=';', index=False, encoding='utf-8-sig')

print("\n📊 Amostra da Camada Bronze:")
display(df_bronze.head())

# 2 - Extração das Tabelas de Dimensão (De-Para)

In [ ]:
# ==========================================
# 2 - Extração de Dimensões (27 - Brasil)
# ==========================================
import os
import requests
import pandas as pd

token = obter_access_token()
headers = {**HEADERS_PADRAO, "Authorization": f"Bearer {token}"}

print("⏳ Extraindo tabelas de dimensão da Filial 27 (Brasil)...")

# 1. Setores
res_setores = requests.get(f"https://api.mktzap.com.br/sector/{COMPANY_ID}", headers=headers, verify=False)
dict_setores = {str(item['id']): item['name'] for item in res_setores.json()} if res_setores.status_code == 200 else {}

# 2. Status
res_status = requests.get(f"https://api.mktzap.com.br/status/{COMPANY_ID}", headers=headers, verify=False)
dict_status = {str(item['id']): item['name'] for item in res_status.json()} if res_status.status_code == 200 else {}

# 3. Usuários
res_usuarios = requests.get(f"https://api.mktzap.com.br/user/{COMPANY_ID}", headers=headers, verify=False)
dict_usuarios = {str(item['id']): item['name'] for item in res_usuarios.json()} if res_usuarios.status_code == 200 else {}

# 4. Canais
res_canais = requests.get(f"https://api.mktzap.com.br/company-channels/{COMPANY_ID}", headers=headers, verify=False)
dict_canais = {str(item['id']): item['name'] for item in res_canais.json()} if res_canais.status_code == 200 else {}

# Salvando na estrutura de pastas padronizada de Dimensões
pasta_dim_27 = os.path.join(os.getcwd(), "documentos", "dimensoes", "27 - Brasil")
os.makedirs(pasta_dim_27, exist_ok=True)

pd.DataFrame(list(dict_setores.items()), columns=['id', 'name']).to_csv(
    os.path.join(pasta_dim_27, "dim_setores.csv"), sep=';', index=False, encoding='utf-8-sig'
)
pd.DataFrame(list(dict_status.items()), columns=['id', 'name']).to_csv(
    os.path.join(pasta_dim_27, "dim_status.csv"), sep=';', index=False, encoding='utf-8-sig'
)
pd.DataFrame(list(dict_usuarios.items()), columns=['id', 'name']).to_csv(
    os.path.join(pasta_dim_27, "dim_usuarios.csv"), sep=';', index=False, encoding='utf-8-sig'
)
pd.DataFrame(list(dict_canais.items()), columns=['id', 'name']).to_csv(
    os.path.join(pasta_dim_27, "dim_canais.csv"), sep=';', index=False, encoding='utf-8-sig'
)

print(f"✅ Dimensões da Filial 27 salvas com sucesso em: {pasta_dim_27}")

# 3 - Analítico - Extração do Histórico (Camada Bronze)

In [ ]:
# ==========================================
# 3 - Analítico - Extração do Histórico (Camada Bronze)
# ==========================================
from datetime import datetime, timedelta
import pandas as pd
import os
import requests

def extrair_historico(data_inicio, data_fim):
    token = obter_access_token()
    pagina = 1
    todos_atendimentos = []
    registros_anteriores = None
    url = f"https://api.mktzap.com.br/company/{COMPANY_ID}/historycontact"
    params = {"createdFrom": data_inicio, "createdTo": data_fim}

    print(f"⏳ Extraindo histórico de {data_inicio} até {data_fim}...")
    while True:
        if pagina > 500: break
        headers = {**HEADERS_PADRAO, "Authorization": f"Bearer {token}", "x-page": str(pagina)}
        res = requests.get(url, headers=headers, params=params, verify=False, timeout=30)
        
        if res.status_code == 401:
            token = obter_access_token()
            continue
            
        dados = res.json()
        if not dados or dados == registros_anteriores: break
        
        registros_anteriores = dados
        todos_atendimentos.extend(dados)
        if pagina % 10 == 0: print(f"  -> {pagina} páginas processadas...")
        pagina += 1
        
    return todos_atendimentos

# Lógica de Datas D-1
hoje = datetime.now()
ontem = hoje - timedelta(days=1)

primeiro_dia = ontem.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
ultimo_dia = ontem.replace(hour=23, minute=59, second=59, microsecond=0)

created_from = primeiro_dia.strftime("%Y-%m-%dT%H:%M:%S-03:00")
created_to = ultimo_dia.strftime("%Y-%m-%dT%H:%M:%S-03:00")

dados_brutos = extrair_historico(created_from, created_to)
df_bronze = pd.DataFrame(dados_brutos)
print(f"✅ Extração bruta concluída! Total: {len(df_bronze)} registros.")

if not df_bronze.empty:
    pasta_mes = ontem.strftime("%Y%m")
    pasta_27 = os.path.join(os.getcwd(), "documentos", pasta_mes, "27 - Brasil")
    os.makedirs(pasta_27, exist_ok=True)
    
    nome_bruto = f"historico_atendimentos_periodo_{primeiro_dia.strftime('%Y%m%d')}_a_{ultimo_dia.strftime('%Y%m%d')}.csv"
    caminho_bruto = os.path.join(pasta_27, nome_bruto)
    df_bronze.to_csv(caminho_bruto, sep=';', index=False, encoding='utf-8-sig')

print("\n📊 Amostra da Camada Bronze:")
display(df_bronze.head())

# 4 - Analítico - Transformação (Camada Gold) e Organização de Arquivos

In [ ]:
# ==========================================
# 4 - Analítico - Transformação (27 - Brasil) (Camada Gold)
# ==========================================
import pandas as pd
import os
import ast
import glob
from datetime import datetime, timedelta

SETORES_PERMITIDOS_IDS = ['135', '8488', '8490', '11267', '16273', '16274', '16275', '30798', '34324', '36245', '37067', '37965', '37966', '38932']

print("⏳ A iniciar Transformação Gold do Relatório Analítico (Brasil) com Pandas...")

def carregar_dict_dimensao(nome_arquivo):
    caminho = os.path.join(os.getcwd(), "documentos", "dimensoes", "27 - Brasil", nome_arquivo)
    if os.path.exists(caminho):
        df_dim = pd.read_csv(caminho, sep=';', dtype=str)
        if 'id' in df_dim.columns and 'name' in df_dim.columns:
            return df_dim.set_index('id')['name'].to_dict()
    return {}

dict_setores = carregar_dict_dimensao("dim_setores.csv")
dict_status = carregar_dict_dimensao("dim_status.csv")
dict_usuarios = carregar_dict_dimensao("dim_usuarios.csv")
dict_canais = carregar_dict_dimensao("dim_canais.csv")

df_gold = df_bronze.copy()
df_gold['sector_id'] = df_gold['sector_id'].astype(str)
df_gold = df_gold[df_gold['sector_id'].isin(SETORES_PERMITIDOS_IDS)].copy()

def ajustar_fuso(serie):
    return pd.to_datetime(serie).dt.tz_localize(None) - pd.Timedelta(hours=3)

df_gold['Data Contato'] = ajustar_fuso(df_gold['created_at']).dt.strftime("%d/%m/%Y %H:%M:%S")
df_gold['Encerrado em'] = ajustar_fuso(df_gold['closed_at']).dt.strftime("%d/%m/%Y %H:%M:%S")

def parse_contacts(lista_contatos):
    res = {}
    try:
        if isinstance(lista_contatos, str):
            lista_contatos = ast.literal_eval(lista_contatos)
        if isinstance(lista_contatos, list):
            for c in lista_contatos:
                if isinstance(c, dict) and 'label' in c:
                    res[str(c['label']).strip()] = str(c.get('value', '')).strip()
    except Exception:
        pass
    return res

# Garante que os índices se mantêm alinhados com o df_gold após a filtragem
df_custom = pd.DataFrame([parse_contacts(c) for c in df_gold['contacts']], index=df_gold.index)

# Consolidação de variações no nome da coluna (Qtd Atendimento / Qtd Atendimentos)
cols_qtd = [c for c in df_custom.columns if 'qtd' in c.lower() and 'atend' in c.lower()]

if cols_qtd:
    s_qtd_raw = df_custom[cols_qtd].bfill(axis=1).iloc[:, 0]
else:
    s_qtd_raw = pd.Series("", index=df_gold.index)

def aplicar_regra_qtd(val):
    if pd.isna(val):
        return "1"
    val_str = str(val).strip()
    if val_str == "" or val_str.lower() in ["none", "nan", "null"]:
        return "1"
    try:
        num = float(val_str.replace(',', '.'))
        # Regra de negócio: <= 0 ou >= 100 vira 1.
        if num <= 0 or num >= 100:
            return "1"
        return str(int(num))
    except (ValueError, TypeError):
        # Textos e formatos inválidos viram 1.
        return "1"

df_final = pd.DataFrame()
df_final['Protocolo'] = df_gold['protocol']
df_final['Empresa'] = "Natura - CN"
df_final['Nome'] = df_gold['name'].str.strip('"')
df_final['Data Contato'] = df_gold['Data Contato']
df_final['Setor Fim'] = df_gold['sector_id'].map(dict_setores).fillna(df_gold['sector_id'])
df_final['Status de finalização'] = df_gold['status_id'].astype(str).map(dict_status).fillna(df_gold['status_id'])
df_final['Responsável'] = df_gold['closed_by_user_id'].astype(str).map(dict_usuarios).fillna(df_gold['closed_by_user_id'])
df_final['Canal de Entrada'] = df_gold['channelable_id'].astype(str).map(dict_canais).fillna(df_gold['channelable_id'])
df_final['Qtde Mensagens'] = df_gold['messages_count']
df_final['Encerrado em'] = df_gold['Encerrado em']

df_final['Tempo de Espera_Min'] = pd.to_numeric(df_gold['waiting_time_medium'], errors='coerce') / 60
df_final['Tempo de Espera_Min'] = df_final['Tempo de Espera_Min'].round(2)

colunas_custom_necessarias = [
    "CODIGO", "GV", "PEDIDO FINALIZADO", "MERCADO", 
    "Função do Sistema", "Ciclo Vigente", "Está em Fechamento?", "Código do Impactado", "Ciclo - Health Check"
]

for col in colunas_custom_necessarias:
    df_final[col] = df_custom[col] if col in df_custom.columns else ""

# Coluna RAW com o dado original da API e a Coluna oficial com a regra de negócio aplicada
df_final['Qtd Atendimentos RAW'] = s_qtd_raw
df_final['Qtd Atendimentos'] = s_qtd_raw.apply(aplicar_regra_qtd)

pasta_mes = ontem.strftime("%Y%m")
caminho_pasta = os.path.join(os.getcwd(), "documentos", pasta_mes, "27 - Brasil")
os.makedirs(caminho_pasta, exist_ok=True)

data_exec = hoje.strftime('%Y-%m-%d')
data_ini_str = primeiro_dia.strftime('%Y-%m-%d')
data_fim_str = ultimo_dia.strftime('%Y-%m-%d')

nome_saida = f"{data_exec}-relatorio-gold-analitico-periodo-{data_ini_str}-a-{data_fim_str}.csv"
caminho_saida = os.path.join(caminho_pasta, nome_saida)

df_final.fillna("").to_csv(caminho_saida, sep=';', index=False, encoding='utf-8-sig')

# Limpeza dos ficheiros brutos na pasta de destino
arquivos_brutos = glob.glob(os.path.join(caminho_pasta, "historico_atendimentos_periodo_*.csv"))
for arq in arquivos_brutos: 
    try: os.remove(arq)
    except Exception: pass

print(f"✅ Transformação via Pandas concluída! Relatório Gold Analítico salvo em: {caminho_saida}")
print("\n📊 Amostra do Relatório Final Analítico:")
display(df_final.head())

# 5 - Enquete - Extração (Camada Bronze)

In [ ]:
# ==========================================
# 5 - Enquete - Extração (Camada Bronze)
# ==========================================
import pandas as pd
import os
import requests
import time
from datetime import datetime, timedelta

def buscar_id_enquete(nome_enquete):
    token = obter_access_token()
    url = f"https://api.mktzap.com.br/poll/{COMPANY_ID}"
    headers = {**HEADERS_PADRAO, "Authorization": f"Bearer {token}"}
    
    print(f"⏳ Buscando ID da enquete '{nome_enquete}'...")
    res = requests.get(url, headers=headers, verify=False, timeout=30)
    
    if res.status_code == 200:
        enquetes = res.json()
        if isinstance(enquetes, dict) and "data" in enquetes: enquetes = enquetes["data"]
        for enq in enquetes:
            if str(enq.get("name")).strip() == nome_enquete:
                print(f"✅ Enquete encontrada! ID: {enq.get('id')}")
                return enq.get("id")
    return None

def extrair_detalhes_enquete(poll_id, data_inicio, data_fim):
    token = obter_access_token()
    pagina = 1
    todos_registros = []
    registros_anteriores = None
    url = f"https://api.mktzap.com.br/poll-answer/{COMPANY_ID}" 
    params = {"poll_id": poll_id, "createdFrom": data_inicio, "createdTo": data_fim}

    print(f"⏳ Extraindo detalhes das enquetes de {data_inicio} até {data_fim}...")
    while True:
        if pagina > 500: break
        headers = {**HEADERS_PADRAO, "Authorization": f"Bearer {token}", "x-page": str(pagina)}
        
        res = None
        for tentativa in range(3):
            res = requests.get(url, headers=headers, params=params, verify=False, timeout=30)
            if res.status_code in [500, 502, 503, 504]: time.sleep(3)
            else: break
                
        if res.status_code == 401:
            token = obter_access_token()
            continue
            
        if res.status_code != 200: break

        dados = res.json()
        if not dados or dados == registros_anteriores: break
        registros_anteriores = dados
        
        lista_dados = []
        if isinstance(dados, list):
            lista_dados = dados[0]["result"] if (len(dados) == 1 and isinstance(dados[0], dict) and "result" in dados[0]) else dados
        elif isinstance(dados, dict):
            lista_dados = dados.get("result", dados.get("data", []))
                
        if not lista_dados: break
        todos_registros.extend(lista_dados)
        if pagina % 5 == 0: print(f"  -> {pagina} páginas processadas...")
        pagina += 1
        
    return todos_registros

hoje = datetime.now()
ontem = hoje - timedelta(days=1)
primeiro_dia = ontem.replace(day=1)
ultimo_dia = ontem

created_from = primeiro_dia.strftime("%Y-%m-%d")
created_to = ultimo_dia.strftime("%Y-%m-%d")

NOME_ENQUETE = "C-SAT (Experiência CNF)"
poll_id_alvo = buscar_id_enquete(NOME_ENQUETE)

dados_enquetes = extrair_detalhes_enquete(poll_id_alvo, created_from, created_to) if poll_id_alvo else []
df_bronze_enquetes = pd.DataFrame(dados_enquetes)

print(f"✅ Extração bruta concluída! Total: {len(df_bronze_enquetes)} enquetes detalhadas.")

if not df_bronze_enquetes.empty:
    pasta_mes = ontem.strftime("%Y%m")
    pasta_27 = os.path.join(os.getcwd(), "documentos", pasta_mes, "27 - Brasil")
    os.makedirs(pasta_27, exist_ok=True)
    
    nome_bruto_enq = f"bruto_enquetes_detalhado_{primeiro_dia.strftime('%Y%m%d')}_a_{ultimo_dia.strftime('%Y%m%d')}.csv"
    caminho_bruto_enq = os.path.join(pasta_27, nome_bruto_enq)
    df_bronze_enquetes.to_csv(caminho_bruto_enq, sep=';', index=False, encoding='utf-8-sig')
    print("\n📊 Amostra da Camada Bronze (Enquetes Detalhadas):")
    display(df_bronze_enquetes.head())
else:
    raise ValueError("⚠️ Nenhuma resposta recebida no período. Verifique os parâmetros.")

# 6 - Enquete - Transformação (Camada Gold) e Organização

In [ ]:
# ==========================================
# 6 - Enquete - Transformação (Camada Gold)
# ==========================================
import pandas as pd
import os
import glob

print("⏳ Iniciando Transformação Gold do Relatório Detalhado de Enquetes...")

df_enq = df_bronze_enquetes.copy()

if df_enq.empty or 'answer' not in df_enq.columns:
    raise ValueError("⚠️ A base bruta de enquetes está vazia ou incorreta.")

def ajustar_fuso_dt(serie):
    dt = pd.to_datetime(serie, format='%Y-%m-%d %H:%M:%S', errors='coerce')
    return dt - pd.Timedelta(hours=3)

df_enq['dt_contato_brt'] = ajustar_fuso_dt(df_enq['created_at'])
df_enq['dt_enviado_brt'] = ajustar_fuso_dt(df_enq['sent_at'])
df_enq['dt_respondido_brt'] = ajustar_fuso_dt(df_enq['answered_at'])

limite_inicio = pd.to_datetime(primeiro_dia.strftime("%Y-%m-%d 00:00:00"))
limite_fim = pd.to_datetime(ultimo_dia.strftime("%Y-%m-%d 23:59:59"))

df_enq = df_enq[(df_enq['dt_enviado_brt'] >= limite_inicio) & (df_enq['dt_enviado_brt'] <= limite_fim)].copy()

if df_enq.empty:
    raise ValueError("⚠️ Nenhum registro encontrado na janela D-1 para a data de envio ('Enviado em').")

df_gold_detalhado = pd.DataFrame()
df_gold_detalhado['Data do Contato'] = df_enq['dt_contato_brt'].dt.strftime('%Y-%m-%d %H:%M:%S').fillna('Não respondido')
df_gold_detalhado['Enviado em'] = df_enq['dt_enviado_brt'].dt.strftime('%Y-%m-%d %H:%M:%S').fillna('Não respondido')
df_gold_detalhado['Respondido em'] = df_enq['dt_respondido_brt'].dt.strftime('%Y-%m-%d %H:%M:%S').fillna('Não respondido')
df_gold_detalhado['Protocolo'] = df_enq['protocol']
df_gold_detalhado['Nome'] = df_enq['name']

if 'df_bronze' in globals() and not df_bronze.empty and 'protocol' in df_bronze.columns:
    dict_canais = carregar_dict_dimensao("dim_canais.csv")
    df_atend_lookup = df_bronze[['protocol', 'channelable_id', 'contact_id']].drop_duplicates('protocol').copy()
    df_atend_lookup['Canal'] = df_atend_lookup['channelable_id'].astype(str).map(dict_canais).fillna("")
    
    df_gold_detalhado = df_gold_detalhado.merge(
        df_atend_lookup[['protocol', 'Canal', 'contact_id']], 
        left_on='Protocolo', right_on='protocol', how='left'
    )
    df_gold_detalhado['Contato'] = df_gold_detalhado['contact_id'].fillna("")
    df_gold_detalhado.drop(columns=['protocol', 'contact_id'], inplace=True, errors='ignore')
else:
    df_gold_detalhado['Canal'] = ""
    df_gold_detalhado['Contato'] = ""

df_gold_detalhado['Responsável'] = df_enq['user_name']
df_gold_detalhado['Nota'] = df_enq['answer'].replace({'Não avaliado': 'Não respondido'}).fillna('Não respondido')

colunas_finais = ['Data do Contato', 'Enviado em', 'Respondido em', 'Protocolo', 'Nome', 'Canal', 'Contato', 'Responsável', 'Nota']
df_gold_detalhado = df_gold_detalhado.reindex(columns=colunas_finais)

pasta_mes = ontem.strftime("%Y%m")
caminho_pasta = os.path.join(os.getcwd(), "documentos", pasta_mes, "27 - Brasil")
os.makedirs(caminho_pasta, exist_ok=True)

data_exec = hoje.strftime('%Y-%m-%d')
data_ini_str = primeiro_dia.strftime('%Y-%m-%d')
data_fim_str = ultimo_dia.strftime('%Y-%m-%d')

nome_saida_enq = f"{data_exec}-relatorio-gold-enquetes-detalhado-{data_ini_str}-a-{data_fim_str}.csv"
caminho_saida_enq = os.path.join(caminho_pasta, nome_saida_enq)

df_gold_detalhado.to_csv(caminho_saida_enq, sep=';', index=False, encoding='utf-8-sig')

arquivos_brutos_enq = glob.glob(os.path.join(caminho_pasta, "bruto_enquetes_*.csv"))
for arq in arquivos_brutos_enq: 
    try: os.remove(arq)
    except Exception: pass

print(f"✅ Transformação concluída! Relatório Gold salvo em: {caminho_saida_enq}")
print(f"📊 Registros validados no intervalo 'Enviado em' ({data_ini_str} 00:00:00 a {data_fim_str} 23:59:59): {len(df_gold_detalhado)}")
print("\n📊 Amostra do Relatório Final Detalhado:")
display(df_gold_detalhado.head())

# 7 - Relatório Completo - Extração (Camada Bronze via API)

In [ ]:
# ==========================================
# 7 - Relatório Completo - Extração (Camada Bronze)
# ==========================================
import pandas as pd
import os
import requests
import time
from datetime import datetime, timedelta

def extrair_historico_completo_api(data_inicio_iso, data_fim_iso):
    token = obter_access_token()
    pagina = 1
    todos_registros = []
    registros_anteriores = None
    url = f"https://api.mktzap.com.br/company/{COMPANY_ID}/history"
    params = {"updatedFrom": data_inicio_iso, "updatedTo": data_fim_iso, "showReferralTracking": "true"}

    print(f"⏳ Extraindo histórico de encerramentos via API de {data_inicio_iso} até {data_fim_iso}...")
    while True:
        if pagina > 1000: break
        headers = {**HEADERS_PADRAO, "Authorization": f"Bearer {token}", "x-page": str(pagina)}
        
        res = None
        for tentativa in range(3):
            res = requests.get(url, headers=headers, params=params, verify=False, timeout=30)
            if res.status_code in [500, 502, 503, 504]: time.sleep(3)
            else: break
                
        if res.status_code == 401:
            token = obter_access_token()
            continue
            
        if res.status_code != 200: break

        dados = res.json()
        if not dados or dados == registros_anteriores: break
        registros_anteriores = dados
        
        lista_dados = dados if isinstance(dados, list) else dados.get("data", [])
        if not lista_dados: break
        
        todos_registros.extend(lista_dados)
        if pagina % 10 == 0: print(f"  -> {pagina} páginas processadas ({len(todos_registros)} registros)...")
        pagina += 1
        
    return todos_registros

hoje = datetime.now()
ontem = hoje - timedelta(days=1)
primeiro_dia = ontem.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
ultimo_dia = ontem.replace(hour=23, minute=59, second=59, microsecond=0)

created_from = primeiro_dia.strftime("%Y-%m-%dT%H:%M:%S-03:00")
created_to = ultimo_dia.strftime("%Y-%m-%dT%H:%M:%S-03:00")

dados_completo = extrair_historico_completo_api(created_from, created_to)
df_bronze_completo = pd.DataFrame(dados_completo)

print(f"✅ Extração bruta concluída! Total: {len(df_bronze_completo)} registros recebidos da API.")

if not df_bronze_completo.empty:
    pasta_mes = ontem.strftime("%Y%m")
    pasta_27 = os.path.join(os.getcwd(), "documentos", pasta_mes, "27 - Brasil")
    os.makedirs(pasta_27, exist_ok=True)
    
    nome_bruto_comp = f"bruto_completo_{primeiro_dia.strftime('%Y%m%d')}_a_{ultimo_dia.strftime('%Y%m%d')}.csv"
    caminho_bruto_comp = os.path.join(pasta_27, nome_bruto_comp)
    df_bronze_completo.to_csv(caminho_bruto_comp, sep=';', index=False, encoding='utf-8-sig')
    print("\n📊 Amostra da Camada Bronze (Relatório Completo):")
    display(df_bronze_completo.head(3))

# 8 - Relatório Completo - Transformação (Camada Gold)

In [ ]:
# ==========================================
# 8 - Relatório Completo - Transformação (27 - Brasil) (Camada Gold)
# ==========================================
import pandas as pd
import os
import glob
from datetime import datetime, timedelta

print("⏳ Iniciando Transformação Gold do Relatório Completo (Brasil)...")

SETORES_ALVO_NOMES = ['APP AVON CF', 'APP NATURA', 'EXPERIÊNCIA CB', 'NATURA EXPCB']

def carregar_dict_dimensao(nome_arquivo):
    caminho = os.path.join(os.getcwd(), "documentos", "dimensoes", "27 - Brasil", nome_arquivo)
    if os.path.exists(caminho):
        df_dim = pd.read_csv(caminho, sep=';', dtype=str)
        if 'id' in df_dim.columns and 'name' in df_dim.columns:
            return df_dim.set_index('id')['name'].to_dict()
    return {}

dict_setores = carregar_dict_dimensao("dim_setores.csv")
dict_status = carregar_dict_dimensao("dim_status.csv")
dict_usuarios = carregar_dict_dimensao("dim_usuarios.csv")
dict_canais = carregar_dict_dimensao("dim_canais.csv")

if 'df_bronze_completo' not in globals() or df_bronze_completo.empty:
    print("⚠️ A base 'df_bronze_completo' está vazia ou não foi carregada na Célula 7.")
else:
    df_comp = df_bronze_completo.copy()
    total_inicial = len(df_comp)
    print(f"📊 Total de registros na Camada Bronze (Célula 7): {total_inicial}")

    def formatar_id_str(serie):
        s_num = pd.to_numeric(serie, errors='coerce')
        res = pd.Series("", index=serie.index)
        valid_mask = s_num.notna()
        res[valid_mask] = s_num[valid_mask].astype(int).astype(str)
        return res

    def ajustar_fuso_dt(serie):
        s_dt = pd.to_datetime(serie, errors='coerce', format='ISO8601')
        if s_dt.isna().all() and not pd.isna(serie).all():
            s_dt = pd.to_datetime(serie, errors='coerce')
        
        if hasattr(s_dt.dt, 'tz') and s_dt.dt.tz is not None:
            s_dt = s_dt.dt.tz_convert('America/Sao_Paulo').dt.tz_localize(None)
        else:
            s_dt = s_dt - pd.Timedelta(hours=3)
        return s_dt

    def calcular_tempo_operacional(dt_inicio, dt_fim):
        try:
            if pd.isna(dt_inicio) or pd.isna(dt_fim): return ""
            seg = (dt_fim - dt_inicio).total_seconds()
            if seg < 0 or pd.isna(seg): return ""
            horas = int(seg // 3600)
            minutos = int((seg % 3600) // 60)
            segs = int(seg % 60)
            return f"{horas:02d}:{minutos:02d}:{segs:02d}"
        except Exception:
            return ""

    # Mapeamento e Diagnóstico de Setores
    sector_id_clean = formatar_id_str(df_comp['sector_id'])
    df_comp['Setor_Nome'] = sector_id_clean.map(dict_setores).fillna("")
    
    setores_presentes = df_comp['Setor_Nome'].unique().tolist()
    print(f"🔎 Setores encontrados na base bruta: {setores_presentes}")

    df_comp = df_comp[df_comp['Setor_Nome'].isin(SETORES_ALVO_NOMES)].copy()
    total_apos_setores = len(df_comp)
    print(f"📊 Registros após filtrar Setores Alvo {SETORES_ALVO_NOMES}: {total_apos_setores}")

    # Ajuste e Filtro de Datas
    df_comp['dt_criacao_brt'] = ajustar_fuso_dt(df_comp['created_at'])
    df_comp['dt_conclusao_brt'] = ajustar_fuso_dt(df_comp['closed_at'])

    limite_inicio = pd.to_datetime(primeiro_dia.strftime("%Y-%m-%d 00:00:00"))
    limite_fim = pd.to_datetime(ultimo_dia.strftime("%Y-%m-%d 23:59:59"))

    df_gold_comp = df_comp[(df_comp['dt_conclusao_brt'] >= limite_inicio) & (df_comp['dt_conclusao_brt'] <= limite_fim)].copy()
    total_final = len(df_gold_comp)
    print(f"📊 Registros finais após filtro de conclusão ({primeiro_dia.strftime('%Y-%m-%d')} a {ultimo_dia.strftime('%Y-%m-%d')}): {total_final}")

    if df_gold_comp.empty:
        print(f"ℹ️ Nenhum registro encerrou no período solicitado para os setores configurados.")
    else:
        df_final_completo = pd.DataFrame()
        df_final_completo['Protocol'] = df_gold_comp['protocol']
        df_final_completo['Nome'] = df_gold_comp['name'].str.strip('"')

        df_final_completo['Data contato'] = df_gold_comp['dt_criacao_brt'].dt.strftime("%Y-%m-%d %H:%M:%S").fillna("")
        df_final_completo['Data criação'] = df_final_completo['Data contato']
        df_final_completo['Início Atendimento'] = df_final_completo['Data contato']
        df_final_completo['Atendido'] = ajustar_fuso_dt(df_gold_comp['first_operator_answer_at']).dt.strftime("%Y-%m-%d %H:%M:%S").fillna("")
        df_final_completo['Data conclusão'] = df_gold_comp['dt_conclusao_brt'].dt.strftime("%Y-%m-%d %H:%M:%S").fillna("")
        df_final_completo['Última mensagem'] = ajustar_fuso_dt(df_gold_comp['last_message_at']).dt.strftime("%Y-%m-%d %H:%M:%S").fillna("")

        df_final_completo['Tipo Canal'] = df_gold_comp['channelable_type'].astype(str).replace({'App\\\\Models\\\\Channels\\\\Email': 'Email', 'App\\\\Models\\\\Channels\\\\Waent': 'Whatsapp Enterprise'}).fillna("Outros")
        df_final_completo['Canal'] = formatar_id_str(df_gold_comp['channelable_id']).map(dict_canais).fillna("")
        df_final_completo['Setor'] = df_gold_comp['Setor_Nome']

        user_closed_str = formatar_id_str(df_gold_comp['closed_by_user_id'])
        user_last_str = formatar_id_str(df_gold_comp['last_user_id'])
        df_final_completo['Operador'] = user_closed_str.map(dict_usuarios).fillna(user_last_str.map(dict_usuarios)).fillna("")

        df_final_completo['Telefone'] = ""
        df_final_completo['Situação'] = df_gold_comp['closed_at'].apply(lambda x: 'Concluído' if pd.notna(x) and str(x) != "" else 'Em aberto')
        df_final_completo['Status'] = formatar_id_str(df_gold_comp['status_id']).map(dict_status).fillna("")

        df_final_completo['Tempo médio até ser atendidos'] = ""
        df_final_completo['Tempo médio primeira espera'] = ""
        df_final_completo['Tempo médio atendimento'] = ""
        df_final_completo['Tempo médio espera'] = ""

        df_final_completo['Tempo médio operacional'] = [
            calcular_tempo_operacional(ini, fim) 
            for ini, fim in zip(df_gold_comp['dt_criacao_brt'], df_gold_comp['dt_conclusao_brt'])
        ]

        df_final_completo['Qtde Mensagens'] = pd.to_numeric(df_gold_comp['messages_count'], errors='coerce').fillna(0).astype(int)

        if 'df_final' in globals() and not df_final.empty and 'Protocolo' in df_final.columns and 'Qtd Atendimentos' in df_final.columns:
            dict_qtd_atend = df_final.set_index('Protocolo')['Qtd Atendimentos'].to_dict()
            df_final_completo['Qtd Atendimento'] = df_final_completo['Protocol'].map(dict_qtd_atend).fillna("1")
        else:
            df_final_completo['Qtd Atendimento'] = "1"

        colunas_oficiais_completo = [
            'Protocol', 'Nome', 'Data contato', 'Data criação', 'Início Atendimento', 'Atendido', 'Data conclusão',
            'Última mensagem', 'Tipo Canal', 'Canal', 'Setor', 'Operador', 'Telefone', 'Situação', 'Status',
            'Tempo médio até ser atendidos', 'Tempo médio primeira espera', 'Tempo médio atendimento',
            'Tempo médio espera', 'Tempo médio operacional', 'Qtde Mensagens', 'Qtd Atendimento'
        ]

        df_final_completo = df_final_completo.reindex(columns=colunas_oficiais_completo)

        pasta_mes = ontem.strftime("%Y%m")
        caminho_pasta = os.path.join(os.getcwd(), "documentos", pasta_mes, "27 - Brasil")
        os.makedirs(caminho_pasta, exist_ok=True)

        data_exec = hoje.strftime('%Y-%m-%d')
        data_ini_str = primeiro_dia.strftime('%Y-%m-%d')
        data_fim_str = ultimo_dia.strftime('%Y-%m-%d')

        nome_saida_comp = f"{data_exec}-relatorio-gold-completo-periodo-{data_ini_str}-a-{data_fim_str}.csv"
        caminho_saida_comp = os.path.join(caminho_pasta, nome_saida_comp)

        df_final_completo.to_csv(caminho_saida_comp, sep=';', index=False, encoding='utf-8-sig')

        arquivos_brutos_comp = glob.glob(os.path.join(caminho_pasta, "bruto_completo_*.csv"))
        for arq in arquivos_brutos_comp: 
            try: os.remove(arq)
            except Exception: pass

        print(f"✅ Transformação via API concluída! Relatório Gold salvo em: {caminho_saida_comp}")
        display(df_final_completo.head())

# 9 - Extração das Tabelas de Dimensão (Hispana) (De-Para)

In [ ]:
# ==========================================
# 9 - Extração de Dimensões (1531 - Hispana)
# ==========================================
import os
import requests
import pandas as pd

token_1531 = obter_access_token_1531()
headers_1531 = {**HEADERS_PADRAO, "Authorization": f"Bearer {token_1531}"}

print("⏳ Extraindo tabelas de dimensão da Filial 1531 (Hispana)...")

# 1. Setores
res_setores_1531 = requests.get(f"https://api.mktzap.com.br/sector/{COMPANY_ID_1531}", headers=headers_1531, verify=False)
dict_setores_1531 = {str(item['id']): item['name'] for item in res_setores_1531.json()} if res_setores_1531.status_code == 200 else {}

# 2. Status
res_status_1531 = requests.get(f"https://api.mktzap.com.br/status/{COMPANY_ID_1531}", headers=headers_1531, verify=False)
dict_status_1531 = {str(item['id']): item['name'] for item in res_status_1531.json()} if res_status_1531.status_code == 200 else {}

# 3. Usuários
res_usuarios_1531 = requests.get(f"https://api.mktzap.com.br/user/{COMPANY_ID_1531}", headers=headers_1531, verify=False)
dict_usuarios_1531 = {str(item['id']): item['name'] for item in res_usuarios_1531.json()} if res_usuarios_1531.status_code == 200 else {}

# 4. Canais
res_canais_1531 = requests.get(f"https://api.mktzap.com.br/company-channels/{COMPANY_ID_1531}", headers=headers_1531, verify=False)
dict_canais_1531 = {str(item['id']): item['name'] for item in res_canais_1531.json()} if res_canais_1531.status_code == 200 else {}

# Salvando na estrutura de pastas padronizada de Dimensões
pasta_dim_1531 = os.path.join(os.getcwd(), "documentos", "dimensoes", "1531 - Hispana")
os.makedirs(pasta_dim_1531, exist_ok=True)

pd.DataFrame(list(dict_setores_1531.items()), columns=['id', 'name']).to_csv(
    os.path.join(pasta_dim_1531, "dim_setores.csv"), sep=';', index=False, encoding='utf-8-sig'
)
pd.DataFrame(list(dict_status_1531.items()), columns=['id', 'name']).to_csv(
    os.path.join(pasta_dim_1531, "dim_status.csv"), sep=';', index=False, encoding='utf-8-sig'
)
pd.DataFrame(list(dict_usuarios_1531.items()), columns=['id', 'name']).to_csv(
    os.path.join(pasta_dim_1531, "dim_usuarios.csv"), sep=';', index=False, encoding='utf-8-sig'
)
pd.DataFrame(list(dict_canais_1531.items()), columns=['id', 'name']).to_csv(
    os.path.join(pasta_dim_1531, "dim_canais.csv"), sep=';', index=False, encoding='utf-8-sig'
)

print(f"✅ Dimensões da Filial 1531 salvas com sucesso em: {pasta_dim_1531}")

# 10 - Analítico - Extração do Histórico (Hispana) (Camada Bronze)

In [ ]:
## ==========================================
## 10 - Analítico - Extração do Histórico (Hispana) (Camada Bronze)
## ==========================================
#from datetime import datetime, timedelta
#import pandas as pd
#import os
#import requests
#import time
#
#def extrair_historico_1531(data_inicio, data_fim):
#    token = obter_access_token_1531()
#    pagina = 1
#    todos_atendimentos = []
#    registros_anteriores = None
#    url = f"https://api.mktzap.com.br/company/{COMPANY_ID_1531}/history"
#    params = {"createdFrom": data_inicio, "createdTo": data_fim}
#
#    print(f"⏳ Extraindo histórico bruto (Filial 1531) de {data_inicio} até {data_fim}...")
#    while True:
#        if pagina > 500: break
#        headers = {**HEADERS_PADRAO, "Authorization": f"Bearer {token}", "x-page": str(pagina)}
#        
#        res = None
#        for tentativa in range(3):
#            try:
#                res = requests.get(url, headers=headers, params=params, verify=False, timeout=30)
#                if res.status_code not in [500, 502, 503, 504]: break
#                time.sleep(3)
#            except Exception: time.sleep(3)
#        
#        if res is not None and res.status_code == 401:
#            token = obter_access_token_1531()
#            continue
#            
#        if res is None or res.status_code != 200: break
#            
#        dados = res.json()
#        if not dados or dados == registros_anteriores: break
#        
#        registros_anteriores = dados
#        lista_dados = dados if isinstance(dados, list) else dados.get("data", [])
#        if not lista_dados: break
#        
#        todos_atendimentos.extend(lista_dados)
#        if pagina % 10 == 0: print(f"  -> {pagina} páginas processadas ({len(todos_atendimentos)} registros)...")
#        pagina += 1
#        
#    return todos_atendimentos
#
#hoje = datetime.now()
#ontem = hoje - timedelta(days=1)
#
#primeiro_dia = ontem.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
#ultimo_dia = ontem.replace(hour=23, minute=59, second=59, microsecond=0)
#
#created_from = primeiro_dia.strftime("%Y-%m-%dT%H:%M:%S-03:00")
#created_to = ultimo_dia.strftime("%Y-%m-%dT%H:%M:%S-03:00")
#
#dados_brutos_1531 = extrair_historico_1531(created_from, created_to)
#df_bronze_1531 = pd.DataFrame(dados_brutos_1531)
#print(f"✅ Extração bruta da 1531 concluída! Total: {len(df_bronze_1531)} registros.")
#
#pasta_mes = ontem.strftime("%Y%m")
#pasta_1531 = os.path.join(os.getcwd(), "documentos", pasta_mes, "1531 - Hispana")
#os.makedirs(pasta_1531, exist_ok=True)
#
#if not df_bronze_1531.empty:
#    nome_bruto = f"historico_atendimentos_periodo_{primeiro_dia.strftime('%Y%m%d')}_a_{ultimo_dia.strftime('%Y%m%d')}.csv"
#    caminho_bruto = os.path.join(pasta_1531, nome_bruto)
#    df_bronze_1531.to_csv(caminho_bruto, sep=';', index=False, encoding='utf-8-sig')
#    print(f"📍 Arquivo salvo em: {caminho_bruto}")
#    display(df_bronze_1531.head())
#else:
#    print(f"ℹ️️ Nenhum registro encontrado para a Filial 1531 no período {created_from[:10]} a {created_to[:10]}.")

# 12 - Enquete (1531 - Hispana) - Extração (Camada Bronze)

In [ ]:
# ==========================================
# 12 - Enquete (1531 - Hispana) - Extração (Camada Bronze)
# ==========================================
import pandas as pd
import os
import requests
import time
from datetime import datetime, timedelta

def buscar_id_enquete_1531(nome_enquete=None):
    token = obter_access_token_1531()
    url = f"https://api.mktzap.com.br/poll/{COMPANY_ID_1531}"
    headers = {**HEADERS_PADRAO, "Authorization": f"Bearer {token}"}
    
    print("⏳ Buscando enquetes cadastradas na Filial 1531...")
    res = requests.get(url, headers=headers, verify=False, timeout=30)
    
    if res.status_code == 200:
        enquetes = res.json()
        if isinstance(enquetes, dict) and "data" in enquetes: 
            enquetes = enquetes["data"]
            
        if not enquetes:
            print("⚠️ Nenhuma enquete encontrada na API para a Filial 1531.")
            return None
            
        print(f"📊 Total de enquetes encontradas na 1531: {len(enquetes)}")
        for enq in enquetes:
            print(f"   -> ID: {enq.get('id')} | Nome: '{enq.get('name')}'")
            if nome_enquete and str(enq.get("name")).strip() == nome_enquete:
                print(f"✅ Enquete alvo '{nome_enquete}' encontrada! ID: {enq.get('id')}")
                return enq.get("id")
        
        # Se não informou nome exato ou não encontrou a string, seleciona a primeira enquete disponível
        primeira_enq = enquetes[0]
        print(f"ℹ️ Selecionando enquete ID {primeira_enq.get('id')} ('{primeira_enq.get('name')}') para extração.")
        return primeira_enq.get("id")
        
    print(f"❌ Erro ao consultar enquetes na API: {res.status_code}")
    return None

def extrair_detalhes_enquete_1531(poll_id, data_inicio, data_fim):
    token = obter_access_token_1531()
    pagina = 1
    todos_registros = []
    registros_anteriores = None
    url = f"https://api.mktzap.com.br/poll-answer/{COMPANY_ID_1531}" 
    params = {"poll_id": poll_id, "createdFrom": data_inicio, "createdTo": data_fim}

    print(f"⏳ Extraindo respostas da enquete (1531) de {data_inicio} até {data_fim}...")
    while True:
        if pagina > 500: break
        headers = {**HEADERS_PADRAO, "Authorization": f"Bearer {token}", "x-page": str(pagina)}
        
        res = None
        for tentativa in range(3):
            try:
                res = requests.get(url, headers=headers, params=params, verify=False, timeout=30)
                if res.status_code not in [500, 502, 503, 504]: break
                time.sleep(3)
            except Exception: 
                time.sleep(3)
                
        if res is not None and res.status_code == 401:
            token = obter_access_token_1531()
            continue
            
        if res is None or res.status_code != 200: break

        dados = res.json()
        if not dados or dados == registros_anteriores: break
        registros_anteriores = dados
        
        lista_dados = []
        if isinstance(dados, list):
            lista_dados = dados[0]["result"] if (len(dados) == 1 and isinstance(dados[0], dict) and "result" in dados[0]) else dados
        elif isinstance(dados, dict):
            lista_dados = dados.get("result", dados.get("data", []))
                
        if not lista_dados: break
        todos_registros.extend(lista_dados)
        if pagina % 5 == 0: print(f"  -> {pagina} páginas processadas ({len(todos_registros)} respostas)...")
        pagina += 1
        
    return todos_registros

# Lógica de Datas D-1 (Formato YYYY-MM-DD exigido por este endpoint)
hoje = datetime.now()
ontem = hoje - timedelta(days=1)
primeiro_dia = ontem.replace(day=1)
ultimo_dia = ontem

created_from = primeiro_dia.strftime("%Y-%m-%d")
created_to = ultimo_dia.strftime("%Y-%m-%d")

# Executa busca de ID de enquete
poll_id_alvo_1531 = buscar_id_enquete_1531()

dados_enquetes_1531 = extrair_detalhes_enquete_1531(poll_id_alvo_1531, created_from, created_to) if poll_id_alvo_1531 else []
df_bronze_enquetes_1531 = pd.DataFrame(dados_enquetes_1531)

print(f"✅ Extração bruta de enquetes (1531) concluída! Total: {len(df_bronze_enquetes_1531)} registros.")

pasta_mes = ontem.strftime("%Y%m")
pasta_1531 = os.path.join(os.getcwd(), "documentos", pasta_mes, "1531 - Hispana")
os.makedirs(pasta_1531, exist_ok=True)

if not df_bronze_enquetes_1531.empty:
    nome_bruto_enq = f"bruto_enquetes_detalhado_{primeiro_dia.strftime('%Y%m%d')}_a_{ultimo_dia.strftime('%Y%m%d')}.csv"
    caminho_bruto_enq = os.path.join(pasta_1531, nome_bruto_enq)
    df_bronze_enquetes_1531.to_csv(caminho_bruto_enq, sep=';', index=False, encoding='utf-8-sig')
    print(f"📍 Arquivo salvo em: {caminho_bruto_enq}")
    print("\n📊 Amostra da Camada Bronze (Enquetes Detalhadas 1531):")
    display(df_bronze_enquetes_1531.head())
else:
    print(f"ℹ️ Nenhum registro de enquete retornado para a Filial 1531 no período {created_from} a {created_to}.")

# 13 - Enquete (1531 - Hispana) - Transformação (Camada Gold)

In [ ]:
# ==========================================
# 13 - Enquete (1531 - Hispana) - Transformação (Camada Gold)
# ==========================================
import pandas as pd
import os
import glob
from datetime import datetime, timedelta

print("⏳ Iniciando Transformação Gold do Relatório Detalhado de Enquetes (1531 - Hispana)...")

df_enq_1531 = df_bronze_enquetes_1531.copy()

if df_enq_1531.empty or 'answer' not in df_enq_1531.columns:
    raise ValueError("⚠️ A base bruta de enquetes da 1531 está vazia ou incorreta.")

def carregar_dict_dimensao_1531(nome_arquivo):
    caminho = os.path.join(os.getcwd(), "documentos", "dimensoes", "1531 - Hispana", nome_arquivo)
    if os.path.exists(caminho):
        df_dim = pd.read_csv(caminho, sep=';', dtype=str)
        if 'id' in df_dim.columns and 'name' in df_dim.columns:
            return df_dim.set_index('id')['name'].to_dict()
    return {}

def ajustar_fuso_dt(serie):
    dt = pd.to_datetime(serie, format='%Y-%m-%d %H:%M:%S', errors='coerce')
    return dt - pd.Timedelta(hours=3)

# 1. Conversão dos fusos horários para BRT
df_enq_1531['dt_contato_brt'] = ajustar_fuso_dt(df_enq_1531['created_at'])
df_enq_1531['dt_enviado_brt'] = ajustar_fuso_dt(df_enq_1531['sent_at'])
df_enq_1531['dt_respondido_brt'] = ajustar_fuso_dt(df_enq_1531['answered_at'])

# 2. Regra de Negócio: Filtro estrito de D-1 ancorado em "Enviado em" (BRT)
limite_inicio = pd.to_datetime(primeiro_dia.strftime("%Y-%m-%d 00:00:00"))
limite_fim = pd.to_datetime(ultimo_dia.strftime("%Y-%m-%d 23:59:59"))

df_enq_1531 = df_enq_1531[(df_enq_1531['dt_enviado_brt'] >= limite_inicio) & (df_enq_1531['dt_enviado_brt'] <= limite_fim)].copy()

if df_enq_1531.empty:
    raise ValueError("⚠️ Nenhum registro encontrado na janela D-1 para a data de envio ('Enviado em').")

# 3. Montagem do DataFrame Gold oficial
df_gold_detalhado_1531 = pd.DataFrame()
df_gold_detalhado_1531['Data do Contato'] = df_enq_1531['dt_contato_brt'].dt.strftime('%Y-%m-%d %H:%M:%S').fillna('Não respondido')
df_gold_detalhado_1531['Enviado em'] = df_enq_1531['dt_enviado_brt'].dt.strftime('%Y-%m-%d %H:%M:%S').fillna('Não respondido')
df_gold_detalhado_1531['Respondido em'] = df_enq_1531['dt_respondido_brt'].dt.strftime('%Y-%m-%d %H:%M:%S').fillna('Não respondido')
df_gold_detalhado_1531['Protocolo'] = df_enq_1531['protocol']
df_gold_detalhado_1531['Nome'] = df_enq_1531['name']

# Lookup com histórico/dimensões da 1531 (se disponível)
if 'df_bronze_1531' in globals() and not df_bronze_1531.empty and 'protocol' in df_bronze_1531.columns:
    dict_canais_1531 = carregar_dict_dimensao_1531("dim_canais.csv")
    
    df_atend_lookup = df_bronze_1531[['protocol', 'channelable_id', 'contact_id']].drop_duplicates('protocol').copy()
    df_atend_lookup['Canal'] = df_atend_lookup['channelable_id'].astype(str).map(dict_canais_1531).fillna("")
    
    df_gold_detalhado_1531 = df_gold_detalhado_1531.merge(
        df_atend_lookup[['protocol', 'Canal', 'contact_id']], 
        left_on='Protocolo', 
        right_on='protocol', 
        how='left'
    )
    df_gold_detalhado_1531['Contato'] = df_gold_detalhado_1531['contact_id'].fillna("")
    df_gold_detalhado_1531.drop(columns=['protocol', 'contact_id'], inplace=True, errors='ignore')
else:
    df_gold_detalhado_1531['Canal'] = ""
    df_gold_detalhado_1531['Contato'] = ""

df_gold_detalhado_1531['Responsável'] = df_enq_1531['user_name']
df_gold_detalhado_1531['Nota'] = df_enq_1531['answer'].replace({'Não avaliado': 'Não respondido'}).fillna('Não respondido')

# Reordenação de colunas
colunas_finais = ['Data do Contato', 'Enviado em', 'Respondido em', 'Protocolo', 'Nome', 'Canal', 'Contato', 'Responsável', 'Nota']
df_gold_detalhado_1531 = df_gold_detalhado_1531.reindex(columns=colunas_finais)

# 4. Gravação do arquivo final na estrutura de pastas da Hispana
pasta_mes = ontem.strftime("%Y%m")
caminho_pasta = os.path.join(os.getcwd(), "documentos", pasta_mes, "1531 - Hispana")
os.makedirs(caminho_pasta, exist_ok=True)

data_exec = hoje.strftime('%Y-%m-%d')
data_ini_str = primeiro_dia.strftime('%Y-%m-%d')
data_fim_str = ultimo_dia.strftime('%Y-%m-%d')

nome_saida_enq = f"{data_exec}-relatorio-gold-enquetes-detalhado-{data_ini_str}-a-{data_fim_str}.csv"
caminho_saida_enq = os.path.join(caminho_pasta, nome_saida_enq)

df_gold_detalhado_1531.to_csv(caminho_saida_enq, sep=';', index=False, encoding='utf-8-sig')

# Expulsa os arquivos brutos temporários
arquivos_brutos_enq = glob.glob(os.path.join(caminho_pasta, "bruto_enquetes_*.csv"))
for arq in arquivos_brutos_enq: 
    try: os.remove(arq)
    except Exception: pass

print(f"✅ Transformação concluída! Relatório Gold de Enquetes (1531 - Hispana) salvo em: {caminho_saida_enq}")
print(f"📊 Registros validados no intervalo 'Enviado em' ({data_ini_str} 00:00:00 a {data_fim_str} 23:59:59): {len(df_gold_detalhado_1531)}")
print("\n📊 Amostra do Relatório Final Detalhado (1531 - Hispana):")
display(df_gold_detalhado_1531.head())

# 14 - Relatório Completo (1531 - Hispana) - Extração (Camada Bronze)

In [ ]:
## ==========================================
## 16 - Relatório Completo (1531 - Hispana) - Extração (Camada Bronze)
## ==========================================
#import pandas as pd
#import os
#import requests
#import time
#from datetime import datetime, timedelta
#
#def extrair_historico_completo_1531_api(data_inicio_iso, data_fim_iso):
#    token = obter_access_token_1531()
#    pagina = 1
#    todos_registros = []
#    registros_anteriores = None
#    
#    url = f"https://api.mktzap.com.br/company/{COMPANY_ID_1531}/history"
#    params = {
#        "updatedFrom": data_inicio_iso,
#        "updatedTo": data_fim_iso,
#        "showReferralTracking": "true"
#    }
#
#    print(f"⏳ Extraindo histórico de encerramentos da 1531 via API de {data_inicio_iso} até {data_fim_iso}...")
#    while True:
#        if pagina > 1000: break
#        headers = {**HEADERS_PADRAO, "Authorization": f"Bearer {token}", "x-page": str(pagina)}
#        
#        res = None
#        for tentativa in range(3):
#            try:
#                res = requests.get(url, headers=headers, params=params, verify=False, timeout=30)
#                if res.status_code not in [500, 502, 503, 504]:
#                    break
#                time.sleep(3)
#            except Exception:
#                time.sleep(3)
#                
#        if res is not None and res.status_code == 401:
#            token = obter_access_token_1531()
#            continue
#            
#        if res is None or res.status_code != 200:
#            print(f"❌ Erro na API: {res.status_code if res else 'Timeout'}")
#            break
#
#        dados = res.json()
#        if not dados or dados == registros_anteriores: break
#        registros_anteriores = dados
#        
#        lista_dados = dados if isinstance(dados, list) else dados.get("data", [])
#        if not lista_dados: break
#        
#        todos_registros.extend(lista_dados)
#        if pagina % 10 == 0: 
#            print(f"  -> {pagina} páginas processadas ({len(todos_registros)} registros)...")
#        pagina += 1
#        
#    return todos_registros
#
## Lógica D-1
#hoje = datetime.now()
#ontem = hoje - timedelta(days=1)
#primeiro_dia = ontem.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
#ultimo_dia = ontem.replace(hour=23, minute=59, second=59, microsecond=0)
#
#created_from = primeiro_dia.strftime("%Y-%m-%dT%H:%M:%S-03:00")
#created_to = ultimo_dia.strftime("%Y-%m-%dT%H:%M:%S-03:00")
#
#dados_completo_1531 = extrair_historico_completo_1531_api(created_from, created_to)
#df_bronze_completo_1531 = pd.DataFrame(dados_completo_1531)
#
#print(f"✅ Extração bruta do Relatório Completo (1531) concluída! Total: {len(df_bronze_completo_1531)} registros.")
#
#pasta_1531 = os.path.join(os.getcwd(), "documentos", "1531")
#os.makedirs(pasta_1531, exist_ok=True)
#
#if not df_bronze_completo_1531.empty:
#    nome_bruto_comp = f"bruto_completo_{primeiro_dia.strftime('%Y%m%d')}_a_{ultimo_dia.strftime('%Y%m%d')}.csv"
#    caminho_bruto_comp = os.path.join(pasta_1531, nome_bruto_comp)
#    df_bronze_completo_1531.to_csv(caminho_bruto_comp, sep=';', index=False, encoding='utf-8-sig')
#    print(f"📍 Arquivo salvo em: {caminho_bruto_comp}")
#    print("\n📊 Amostra da Camada Bronze (Relatório Completo 1531):")
#    display(df_bronze_completo_1531.head(3))
#else:
#    print(f"ℹ️ Nenhum registro de encerramento retornado para a 1531 no período {created_from[:10]} a {created_to[:10]}.")

# 16 - Automação Selenium & Gold (Tickets Brasil e Hispana)

In [ ]:
# ==========================================
# Célula Unificada - Automação Selenium & Gold (Tickets Brasil e Hispana)
# ==========================================
import os
import time
import glob
import shutil
import pandas as pd
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 1. Configuração de Datas e Mapeamento de Filiais
hoje = datetime.now()
ontem = hoje - timedelta(days=1)
primeiro_dia = ontem.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
pasta_mes = ontem.strftime("%Y%m")
caminho_base = os.getcwd()

tenants = [
    {
        "id": "27",
        "nome_exibicao": "27 - Brasil",
        "nome_empresa_mktzap": "27 - Natura - CN",
        "pasta_destino": os.path.abspath(os.path.join(caminho_base, "documentos", pasta_mes, "27 - Brasil")),
        "pasta_temp": os.path.abspath(os.path.join(caminho_base, "downloads_temp_27"))
    },
    {
        "id": "1531",
        "nome_exibicao": "1531 - Hispana",
        "nome_empresa_mktzap": "1531 - Natura - Experiencia CN OIs",
        "pasta_destino": os.path.abspath(os.path.join(caminho_base, "documentos", pasta_mes, "1531 - Hispana")),
        "pasta_temp": os.path.abspath(os.path.join(caminho_base, "downloads_temp_1531"))
    }
]

# Garantia da existência dos diretórios
for t in tenants:
    os.makedirs(t["pasta_temp"], exist_ok=True)
    os.makedirs(t["pasta_destino"], exist_ok=True)

MKTZAP_URL_LOGIN = "https://natura.mktzap.com.br/login"
MKTZAP_URL_TICKETS = "https://natura.mktzap.com.br/reports/history-tickets"

# 2. Configuração do Navegador
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-popup-blocking")
chrome_options.add_argument("--disable-features=DownloadBubble,DownloadBubbleV2")

prefs = {
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
    "profile.default_content_settings.popups": 0
}
chrome_options.add_experimental_option("prefs", prefs)

print("🚀 Iniciando navegador unificado do Selenium...")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

try:
    wait = WebDriverWait(driver, 600)  # Aguarda até 10 minutos para o login manual
    wait_curto = WebDriverWait(driver, 20)
    
    # 3. Intervenção Manual Única: Login
    driver.get(MKTZAP_URL_LOGIN)
    print("⏳ Aguardando você realizar o login no MKTZap na janela que abriu...")
    
    wait.until(EC.url_contains("/chat"))
    print("✅ Login realizado com sucesso! Iniciando extração automatizada das filiais...\n")
    time.sleep(3)
    
    # 4. Loop de Extração Sequencial
    for t in tenants:
        nome_empresa = t["nome_empresa_mktzap"]
        pasta_temp = t["pasta_temp"]
        pasta_destino = t["pasta_destino"]
        nome_exibicao = t["nome_exibicao"]
        
        print(f"==================================================")
        print(f"🔄 Processando Filial: {nome_exibicao}")
        print(f"==================================================")
        
        # Define a pasta temporária ativa via CDP
        driver.execute_cdp_cmd("Page.setDownloadBehavior", {
            "behavior": "allow",
            "downloadPath": pasta_temp
        })
        
        # A. Troca de Empresa
        print(f"🏢 Selecionando a empresa '{nome_empresa}' no painel...")
        menu_user = wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "[data-test='menu-user-bottom-a-dropdown']")))
        driver.execute_script("arguments[0].click();", menu_user)
        time.sleep(1)
        
        btn_alterar_empresa = wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "[data-test='menu-user-bottom-a-change-company']")))
        driver.execute_script("arguments[0].click();", btn_alterar_empresa)
        time.sleep(1)
        
        input_empresa = wait_curto.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "[data-test='menu-user-bottom-modal-change-company-text-name']")))
        input_empresa.clear()
        
        for letra in nome_empresa:
            input_empresa.send_keys(letra)
            time.sleep(0.03)
        
        time.sleep(1.5)
        
        item_lista = wait_curto.until(EC.presence_of_element_located((By.XPATH, f"//li[contains(text(), '{nome_empresa}')]")))
        driver.execute_script("arguments[0].click();", item_lista)
        time.sleep(1)
        
        btn_trocar = wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "[data-test='menu-user-botton-modal-change-company-button-ok']")))
        driver.execute_script("arguments[0].click();", btn_trocar)
        
        print("⏳ Aguardando o reload do contexto da empresa...")
        time.sleep(6)
        
        # B. Navegação para o Relatório de Tickets
        print("📍 Acessando o relatório de Tickets...")
        driver.get(MKTZAP_URL_TICKETS)
        wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input[ng-model='vm.filters.createdAtFromDate']")))
        time.sleep(3)
        
        # C. Injeção do Período D-1
        data_de = primeiro_dia.strftime("%d/%m/%Y")
        data_ate = ontem.strftime("%d/%m/%Y")
        print(f"📅 Injetando período D-1: {data_de} até {data_ate}...")
        
        input_from = driver.find_element(By.CSS_SELECTOR, "input[ng-model='vm.filters.createdAtFromDate']")
        input_to = driver.find_element(By.CSS_SELECTOR, "input[ng-model='vm.filters.createdAtToDate']")
        
        driver.execute_script("arguments[0].removeAttribute('readonly'); arguments[0].value = arguments[1];", input_from, data_de)
        driver.execute_script("arguments[0].removeAttribute('readonly'); arguments[0].value = arguments[1];", input_to, data_ate)
        
        driver.execute_script("angular.element(arguments[0]).triggerHandler('change');", input_from)
        driver.execute_script("angular.element(arguments[0]).triggerHandler('change');", input_to)
        
        # D. Disparo da Geração
        print("⚙️ Solicitando geração do relatório...")
        btn_gerar = driver.find_element(By.CSS_SELECTOR, "button[ga-event='mktzap_bt_action_history_tickets_report_generate']")
        driver.execute_script("arguments[0].click();", btn_gerar)
        time.sleep(3)
        
        modais_confirmar = driver.find_elements(By.XPATH, "//button[contains(text(),'Confirmar')]")
        if modais_confirmar and modais_confirmar[0].is_displayed():
            print("⚡ Confirmando modal de background...")
            driver.execute_script("arguments[0].click();", modais_confirmar[0])
            
        # E. Escuta da Pasta de Download
        print("⏳ Aguardando download do arquivo CSV...")
        tempo_limite = 300  
        inicio_espera = time.time()
        arquivo_baixado = None
        
        while time.time() - inicio_espera < tempo_limite:
            arquivos_temp = [f for f in glob.glob(os.path.join(pasta_temp, "*.csv")) if not f.endswith('.crdownload')]
            if arquivos_temp:
                arquivo_baixado = max(arquivos_temp, key=os.path.getmtime)
                if os.path.getmtime(arquivo_baixado) >= inicio_espera - 5:
                    break
            time.sleep(3)
            
        if arquivo_baixado:
            # F. Gravação da Camada Bronze
            nome_padrao_bronze = f"bruto_tickets_{primeiro_dia.strftime('%Y%m%d')}_a_{ontem.strftime('%Y%m%d')}.csv"
            caminho_final_bronze = os.path.join(pasta_destino, nome_padrao_bronze)
            
            if os.path.exists(caminho_final_bronze):
                os.remove(caminho_final_bronze)
                
            shutil.move(arquivo_baixado, caminho_final_bronze)
            print(f"✅ Camada Bronze concluída para {nome_exibicao}!")
            print(f"📍 Arquivo bruto salvo em: {caminho_final_bronze}")
            
            # G. Transformação Imediata para Camada Gold
            print(f"⏳ Gerando Camada Gold para {nome_exibicao}...")
            df_tickets = pd.read_csv(caminho_final_bronze, skiprows=1, sep=None, engine='python', encoding='utf-8-sig')
            
            df_tickets.columns = df_tickets.columns.str.strip()
            for col in df_tickets.select_dtypes(include=['object', 'string']).columns:
                df_tickets[col] = df_tickets[col].astype(str).str.strip().replace({'nan': '', 'None': ''})
                
            data_exec_str = hoje.strftime('%Y-%m-%d')
            data_ini_str = primeiro_dia.strftime('%Y-%m-%d')
            data_fim_str = ontem.strftime('%Y-%m-%d')
            
            nome_gold = f"{data_exec_str}-relatorio-gold-tickets-detalhado-{data_ini_str}-a-{data_fim_str}.csv"
            caminho_gold = os.path.join(pasta_destino, nome_gold)
            
            df_tickets.to_csv(caminho_gold, sep=';', index=False, encoding='utf-8-sig')
            
            # Limpeza do arquivo bruto intermediário
            try:
                os.remove(caminho_final_bronze)
            except Exception:
                pass
                
            print(f"✅ Camada Gold concluída para {nome_exibicao}!")
            print(f"📍 Arquivo Gold salvo em: {caminho_gold}")
            print(f"📊 Total de registros: {len(df_tickets)}\n")
        else:
            print(f"⚠️ Tempo limite expirou para a filial {nome_exibicao}.\n")

finally:
    driver.quit()
    print("🔒 Navegador encerrado com sucesso!")

In [ ]:
# ==========================================
# Célula Final - Limpeza de Diretórios Temporários (Forçada)
# ==========================================
import os
import shutil
import stat
import time

print("🧹 Aguardando liberação de processos (OneDrive/Chrome) para iniciar limpeza...")
time.sleep(3)  # Dá 3 segundos para o Windows e OneDrive soltarem as pastas

def forcar_exclusao(func, path, exc_info):
    """
    Função de callback para remover a trava de 'Somente Leitura' 
    caso o Windows bloqueie a exclusão de algum arquivo interno.
    """
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print(f"⚠️ Aviso ignorado ao excluir {path}: {e}")

caminho_base = os.getcwd()
pastas_para_remover = [
    "downloads_temp",
    "downloads_temp_27",
    "downloads_temp_1531"
]

for pasta in pastas_para_remover:
    caminho_pasta = os.path.join(caminho_base, pasta)
    if os.path.exists(caminho_pasta):
        try:
            # O parâmetro onerror chama a nossa função caso esbarre no Acesso Negado
            shutil.rmtree(caminho_pasta, onerror=forcar_exclusao)
            print(f"✅ Pasta eliminada: {pasta}")
        except Exception as e:
            print(f"⚠️ A pasta '{pasta}' ainda está trancada pelo Windows/OneDrive: {e}")
    else:
        pass 

print("✨ Limpeza do ambiente concluída!")